# Course 5: Building a Small Bangla Language Model
## Session 5: Designing the model

**Goal:** write a small GPT, account for every parameter and pass the sanity checks.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every code cell here was run once on a CPU (November 2026) using the small built-in sample. Cells marked TODO or NOT RUN need your data, a GPU or the network.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

### 1. Count parameters by formula and in code
`params = V*d + T*d + L*(12*d*d + 13*d) + 2*d` with tied output weights.

In [ ]:
def formula(V, d, L, T):
    return V * d + T * d + L * (12 * d * d + 13 * d) + 2 * d

cfg = Config(vocab_size=8000, block_size=256, n_layer=6, n_head=4, d_model=256)
model = GPT(cfg)
in_code = sum(p.numel() for p in model.parameters())
print(in_code, formula(8000, 256, 6, 256), in_code == formula(8000, 256, 6, 256))

### 2. Loss at initialization
It should be close to `ln(vocab_size)`.

In [ ]:
torch.manual_seed(0)
x = torch.randint(0, cfg.vocab_size, (8, 64))
y = torch.randint(0, cfg.vocab_size, (8, 64))
_, loss = model(x, y)
print("initial loss:", round(loss.item(), 3), "| ln V:", round(math.log(cfg.vocab_size), 3))

### 3. Causality test
Changing the last token must not change earlier outputs.

In [ ]:
model.eval()
x2 = x.clone(); x2[:, -1] = (x2[:, -1] + 1) % cfg.vocab_size
with torch.no_grad():
    a, _ = model(x); b, _ = model(x2)
print("earlier positions unchanged:", torch.allclose(a[:, :-1], b[:, :-1], atol=1e-5))
print("last position changed:", not torch.allclose(a[:, -1], b[:, -1], atol=1e-5))

### 4. Overfit one batch
A small model is used here so it finishes quickly on a CPU.

In [ ]:
small = GPT(Config(vocab_size=500, block_size=32, n_layer=2, n_head=2, d_model=64))
xb = torch.randint(0, 500, (8, 32)); yb = torch.randint(0, 500, (8, 32))
opt = torch.optim.AdamW(small.parameters(), lr=3e-3)
for step in range(300):
    _, loss = small(xb, yb)
    opt.zero_grad(); loss.backward(); opt.step()
print("loss after 300 steps on one batch:", round(loss.item(), 4))

### 5. Two shapes at one budget (TODO)
Change the numbers until both are near your target parameter count.

In [ ]:
for L, d in [(6, 256), (12, 180)]:
    print(f"layers {L}, width {d}: {formula(8000, d, L, 256):,} parameters")